In [21]:
import pandas as pd

df=pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,clean_text,predicted_action,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral,reminder the client meeting is scheduled at 10...,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite,your invoice of inr 25515 09 is due on 2025 12...,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,ignore,neutral,reminder the client meeting is scheduled at 11...,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite,hello team please find the attached weekly rep...,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite,hello team please find the attached weekly rep...,respond,neutral


In [22]:
#email_assistant function
def email_assistant(email_text):
    text = email_text.lower()

    urgent_keywords = ["urgent", "submit", "deadline"]
    polite_keywords = ["thanks", "thank you", "appreciate"]

    if any(word in text for word in urgent_keywords):
        return {
            "action": "notify",
            "tone": "urgent"
        }

    if any(word in text for word in polite_keywords):
        return {
            "action": "ignore",
            "tone": "polite"
        }

    return {
        "action": "respond",
        "tone": "neutral"
    }




In [23]:
dangerous_actions = ["respond"] # defined dangerous actions.

# Human in loop
def hitl_check(action):
    if action in dangerous_actions:
       return "Wait_for_Human"
    return "Auto_Approve"

def human_approval():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"
   

In [24]:
results = []

for _, row in df.sample(5).iterrows():
    result = email_assistant(row["body"])
    action = result["action"]
    tone = result["tone"]

    status = hitl_check(action)

    if status == "Wait_for_Human":
        approved = human_approval()
        final_action = action if approved else "blocked"
    else:
        final_action = action

    results.append({
        "email_snippet": row["body"][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })
    

In [25]:
#saving the changes
eval_df = pd.DataFrame(results)
display(eval_df)

eval_df.to_csv("../data/HITL_milestone3_output.csv", index=False)
print("Output saved successfully")

,email_snippet,ai_action,final_action,hitl_status
0,"Hi, don't miss our sale with discounts up to 7...",respond,blocked,Wait_for_Human
1,Congratulations! You have been selected as a l...,respond,blocked,Wait_for_Human
2,"Hello team, please find the attached weekly re...",respond,blocked,Wait_for_Human
3,Security alert: multiple failed login attempts...,respond,blocked,Wait_for_Human
4,Security alert: multiple failed login attempts...,respond,blocked,Wait_for_Human


Output saved successfully
